# 01 — EPC Data Exploration

Before building any pipeline, this notebook documents the raw EPC data: what each column means, data quality issues, distributions of key fields, and the decisions those findings drove in the ingestion step.

**Source**: MHCLG Domestic EPC Open Data — `certificates-*.csv` files covering all London local authorities.

**A row represents**: A single energy performance certificate inspection of one property. Properties can appear multiple times if inspected in different years — we take the most recent per property in the silver layer.

**Note**: The `epc_raw/` folder also contains `recommendations-*.csv` files (suggested improvements per property). Those are analysed separately in `06_recommendations_analysis.ipynb`. This notebook reads certificates only.

In [1]:
import os, glob
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, min, max, round as spark_round, year, to_date

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('explore_epc') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Read certificates only — not recommendations
cert_files = glob.glob('../data/bronze/epc_raw/certificates-*.csv')
print(f'Certificate files found: {len(cert_files)}')

raw = spark.read.csv(cert_files, header=True, inferSchema=False)
print(f'Total rows: {raw.count():,}')
print(f'Total columns: {len(raw.columns)}')

Certificate files found: 15


Total rows: 23,546,857
Total columns: 93


## 1. Column glossary — what does each field mean?

The raw file has ~90 columns. These are the ones we actually use and what they represent:

| Column | What it means |
|---|---|
| `certificate_number` | Unique ID for this EPC inspection. A property can have multiple certificates over time — one per inspection. |
| `address1` / `postcode` | Property address. Postcode is useful for geographic lookup. |
| `local_authority_label` | The London borough (e.g. 'Hackney', 'Barking and Dagenham'). This is how we group by borough. |
| `region` | Government Office Region code. London = `E12000007`. Used to filter to London only. |
| `tenure` | Who lives there and on what basis — `rental (social)`, `rental (private)`, `owner-occupied` etc. We filter to social only. |
| `property_type` | Physical type of the property — Flat, House, Bungalow, Maisonette, Park home. |
| `construction_age_band` | When the property was built — e.g. `England and Wales: before 1900`, `1967-1975`. Older = harder/more expensive to retrofit. |
| `current_energy_rating` | The EPC band: A (best) → G (worst). C is the 2030 government target for social housing. |
| `current_energy_efficiency` | The numeric score behind the rating. 1–100, higher = more efficient. Band C = 69–80. |
| `potential_energy_rating` | What rating the property could achieve if all recommended improvements were made. |
| `potential_energy_efficiency` | Numeric score for the potential rating above. |
| `total_floor_area` | Total floor area in m². Used to normalise CO2 figures. |
| `co2_emiss_curr_per_floor_area` | CO2 emissions per m² of floor area (kg/m²/year). More comparable across properties of different sizes than raw CO2. |
| `main_fuel` | Primary heating fuel — gas, electricity, oil, heat network etc. Relevant for retrofit strategy (electrification vs insulation). |
| `inspection_date` | When the EPC was carried out. Some certificates are 10+ years old. |
| `mains_gas_flag` | Whether the property is connected to mains gas. Properties without gas are often harder/more expensive to heat efficiently. |

## 2. Filter scope — London social rented only

This analysis focuses on social rented properties in London — the tenure and geography relevant to a housing association. The following shows what share of the national EPC dataset that represents.

In [2]:
region_names = {
    'E12000001': 'North East',
    'E12000002': 'North West',
    'E12000003': 'Yorkshire and The Humber',
    'E12000004': 'East Midlands',
    'E12000005': 'West Midlands',
    'E12000006': 'East of England',
    'E12000007': 'London',
    'E12000008': 'South East',
    'E12000009': 'South West',
    'W99999999': 'Wales',
    'L99999999': 'Channel Islands',
    'M99999999': 'Isle of Man',
    'N99999999': 'Northern Ireland',
    'S99999999': 'Scotland',
}

print('=== Rows by region (top 10) ===')
region_df = raw.groupBy('region').count().orderBy('count', ascending=False).limit(10).toPandas()
region_df.insert(1, 'region_name', region_df['region'].map(region_names).fillna('Unknown'))
display(region_df.style.format(thousands=','))

print('=== Rows by tenure ===')
display(raw.groupBy('tenure').count().orderBy('count', ascending=False).limit(10).toPandas().style.format(thousands=','))

=== Rows by region (top 10) ===


,region,region_name,count
0,E12000007,London,"3,524,005"
1,E12000008,South East,"3,518,601"
2,E12000002,North West,"3,049,842"
3,E12000006,East of England,"2,387,180"
4,E12000005,West Midlands,"2,276,411"
5,E12000009,South West,"2,250,580"
6,E12000003,Yorkshire and The Humber,"2,238,823"
7,E12000004,East Midlands,"1,909,041"
8,W99999999,Wales,"1,200,225"
9,E12000001,North East,"1,150,254"


=== Rows by tenure ===


,tenure,count
0,owner-occupied,"8,863,555"
1,rented (private),"3,788,578"
2,rented (social),"3,467,613"
3,Owner-occupied,"2,594,955"
4,unknown,"2,310,577"
5,None,"990,384"
6,Rented (social),"780,305"
7,Rented (private),"631,066"
8,Unknown,"119,819"
9,N/A,3


In [3]:
london_social = raw.filter(
    (col('region') == 'E12000007') &
    (col('tenure') == 'rented (social)')
)
total = london_social.count()
print(f'London social rented rows: {total:,}')
print(f'As % of total dataset: {total / raw.count() * 100:.1f}%')

London social rented rows: 480,655


As % of total dataset: 2.0%


## 3. EPC rating and score distribution

EPC ratings run A–G (A = best, G = worst), based on a numeric SAP (Standard Assessment Procedure) score from 1–100.

| Band | Score range | What it means in practice |
|------|-------------|---------------------------|
| A | 92–100 | Most efficient — very low bills, modern insulation |
| B | 81–91 | High efficiency — well insulated, low running costs |
| C | 69–80 | **2030 social housing target** — reasonable bills |
| D | 55–68 | Average UK stock — higher bills, some heat loss |
| E | 39–54 | Poor — high bills, cold in winter, fuel poverty risk |
| F | 21–38 | Very poor — potentially damp, very high bills |
| G | 1–20 | Least efficient — extremely high bills, health risk |

**Legal consequences:** Under Minimum Energy Efficiency Standards (MEES), landlords cannot let a property rated below E. 
The government proposes raising this to C for social housing by 2030 — meaning any property below band C will be unlettable unless improved. 
This is a direct financial risk for housing associations.

**Health consequences:** Cold homes (typically F/G rated) are linked to excess winter deaths, respiratory illness, and damp. 
Public Health England estimates cold homes contribute to around 10,000 excess winter deaths per year in England.

Properties scoring below 69 (band D or worse) require improvement to meet the 2030 target.

In [4]:
print('=== EPC rating distribution (London social rented) ===')
rating_df = (london_social.groupBy('current_energy_rating')
    .count()
    .orderBy('current_energy_rating')
    .toPandas())
display(rating_df.style.format(thousands=','))

below_c = int(rating_df[rating_df['current_energy_rating'].isin(['D','E','F','G'])]['count'].sum())
print(f"Total: {rating_df['count'].sum():,}")
print(f"D and below: {below_c:,}  ({below_c/total*100:.1f}% of total)")

print('=== EPC score statistics ===')
display(london_social.select(
    spark_round(avg(col('current_energy_efficiency').cast('double')), 1).alias('mean_score'),
    min(col('current_energy_efficiency').cast('double')).alias('min_score'),
    max(col('current_energy_efficiency').cast('double')).alias('max_score'),
).toPandas().style.format('{:,.1f}'))

print(f'Note: 2030 government target requires ALL social housing to reach band C or above.')

=== EPC rating distribution (London social rented) ===


,current_energy_rating,count
0,A,306
1,B,"25,268"
2,C,"260,762"
3,D,"156,892"
4,E,"33,084"
5,F,"3,287"
6,G,"1,056"


Total: 480,655
D and below: 194,319  (40.4% of total)
=== EPC score statistics ===


,mean_score,min_score,max_score
0,68.7,1.0,111.0


Note: 2030 government target requires ALL social housing to reach band C or above.


## 4. Borough breakdown — stock size and average EPC score

In [5]:
from pyspark.sql.functions import sum as spark_sum, when

borough_df = london_social \
    .filter(col('local_authority_label').isNotNull()) \
    .groupBy('local_authority_label') \
    .agg(
        count('*').alias('properties'),
        spark_round(avg('current_energy_efficiency'), 1).alias('avg_epc_score'),
        spark_sum(when(col('current_energy_rating').isin('D','E','F','G'), 1).otherwise(0)).alias('below_c_count'),
    ) \
    .orderBy('avg_epc_score') \
    .toPandas()

borough_df['pct_below_c'] = (borough_df['below_c_count'] / borough_df['properties'] * 100).round(1)

display(borough_df.style.format({
    'properties': '{:,.0f}',
    'avg_epc_score': '{:.1f}',
    'below_c_count': '{:,.0f}',
    'pct_below_c': '{:.1f}%',
}))

,local_authority_label,properties,avg_epc_score,below_c_count,pct_below_c
0,Enfield,"18,320",64.7,"9,753",53.2%
1,Barking and Dagenham,"11,670",66.5,"6,448",55.3%
2,Barnet,"20,164",66.9,"10,253",50.8%
3,Redbridge,"6,891",67.0,"3,570",51.8%
4,Haringey,"14,363",67.2,"6,983",48.6%
5,City of London,399,67.6,191,47.9%
6,Camden,"17,272",67.8,"7,772",45.0%
7,Kensington and Chelsea,"10,406",67.8,"4,648",44.7%
8,Merton,"7,704",67.8,"3,402",44.2%
9,Lambeth,"33,612",67.9,"14,604",43.4%


## 5. Construction age distribution

The `construction_age_band` column has three types of values — important to understand before using it:

- **Standard band strings** (the vast majority): `"England and Wales: 1950-1966"`, `"England and Wales: before 1900"` etc. These are the meaningful values for retrofit analysis.
- **Inspection years as bad data**: `2022`, `2021`, `2020`... Some assessors filled this field with the inspection year instead of the construction age. These properties have no usable age data.
- **Individual construction years** (rare noise): `1970`, `1900` etc. — small counts, likely manual entry errors.

Pre-1950 properties are the key retrofit target: solid walls with no cavity means expensive external or internal solid wall insulation (~£4,000–£14,000 per property vs £500–£1,500 for modern cavity wall fill). The `pct_pre_1950` column in the gold layer captures the three pre-1950 standard bands and any individual years cast below 1950, and ignores the inspection-year noise.

In [6]:
display(london_social.groupBy('construction_age_band') \
    .count() \
    .orderBy('count', ascending=False) \
    .limit(25).toPandas().style.format(thousands=","))

,construction_age_band,count
0,England and Wales: 1950-1966,"101,690"
1,England and Wales: 1967-1975,"73,986"
2,England and Wales: 1900-1929,"72,238"
3,England and Wales: 1930-1949,"67,313"
4,England and Wales: 1976-1982,"32,586"
5,England and Wales: 1983-1990,"27,107"
6,England and Wales: before 1900,"25,709"
7,England and Wales: 1991-1995,"21,514"
8,England and Wales: 1996-2002,"18,325"
9,England and Wales: 2007-2011,"12,908"


## 6. Null rates on key columns

The raw dataset has 93 columns. `key_cols` lists only the 13 selected for the silver layer — the other 80 are dropped at ingestion for one of these reasons:

- **Not needed for retrofit prioritisation**: assessor details, survey methodology flags, administrative codes (UPRN, building reference numbers), floor-level breakdowns, etc.
- **Redundant with selected columns**: multiple address fields where `postcode` suffices; both raw CO2 and per-m² CO2 where we keep only the size-normalised figure.
- **Too sparse to be useful**: several columns covering insulation details, glazing types, and heating controls have null rates above 30–50% in spot-checks — not reliable enough to aggregate at borough level.

The null check below confirms the 13 we *do* keep are usable before committing to the silver schema.

In [7]:
key_cols = [
    'local_authority_label', 'postcode', 'tenure', 'region',
    'property_type', 'construction_age_band', 'current_energy_rating',
    'current_energy_efficiency', 'total_floor_area', 'main_fuel',
    'co2_emiss_curr_per_floor_area', 'inspection_date', 'mains_gas_flag'
]

print(f'Total rows: {total:,}\n')
print(f'{"Column":<35} {"Nulls":>10} {"Null %":>8}')
print('-' * 55)
for c in key_cols:
    if c in london_social.columns:
        nulls = london_social.filter(col(c).isNull() | (col(c) == '')).count()
        print(f'{c:<35} {nulls:>10,} {nulls/total*100:>7.1f}%')
    else:
        print(f'{c:<35} {"(not in dataset)":>19}')

Total rows: 480,655

Column                                   Nulls   Null %
-------------------------------------------------------


local_authority_label                      243     0.1%


postcode                                     0     0.0%


tenure                                       0     0.0%


region                                       0     0.0%


...

total_floor_area                             0     0.0%


main_fuel                                4,347     0.9%


co2_emiss_curr_per_floor_area                0     0.0%


inspection_date                              0     0.0%


mains_gas_flag                           6,108     1.3%


## 7. Inspection date range

How recent is the data? Old certificates may not reflect current condition — a property may have been retrofitted since its last inspection.

In [8]:
display(london_social \
    .withColumn('insp_year', year(to_date(col('inspection_date'), 'yyyy-MM-dd'))) \
    .groupBy('insp_year').count() \
    .orderBy('insp_year') \
    .limit(30).toPandas().style.format(thousands=","))

,insp_year,count
0,"2,013",27
1,"2,014","4,790"
2,"2,015","49,662"
3,"2,016","37,212"
4,"2,017","36,042"
5,"2,018","32,965"
6,"2,019","29,261"
7,"2,020","23,911"
8,"2,021","36,444"
9,"2,022","61,897"


## 8. Property type breakdown

In [9]:
display(london_social.groupBy('property_type').count().orderBy('count', ascending=False).toPandas().style.format(thousands=","))

,property_type,count
0,Flat,"319,768"
1,House,"113,872"
2,Maisonette,"41,434"
3,Bungalow,"5,580"
4,Park home,1


## 9. Main fuel type

Fuel type matters for retrofit strategy. Gas-heated homes need boiler upgrades or heat pumps. Properties on heat networks or electric heating have different upgrade paths.

In [10]:
display(london_social.groupBy('main_fuel').count().orderBy('count', ascending=False).limit(15).toPandas().style.format(thousands=","))

,main_fuel,count
0,mains gas (not community),"361,581"
1,mains gas (community),"61,799"
2,electricity (not community),"46,955"
3,None,"4,347"
4,electricity (community),"1,969"
5,Gas: mains gas,"1,175"
6,To be used only when there is no heating/hot-water system or data is from a community network,"1,071"
7,"Electricity: electricity, unspecified tariff",585
8,LPG (not community),477
9,biomass (community),294


### Finding: `main_fuel` has two data-quality issues — addressed in `03_ingest_epc`

**Schema drift (duplicate labels)**  
The EPC naming convention changed between older and newer certificates. These pairs mean the same thing and must be collapsed:

| Old label | New label |
|---|---|
| `Gas: mains gas` | `mains gas (not community)` |
| `Electricity: electricity, unspecified tariff` | `electricity (not community)` |

**Community vs individual heating**  
`(community)` means a shared district heating system — individual properties on it cannot be retrofitted independently. Any retrofit programme needs building-level intervention. This is a fundamentally different upgrade pathway and must be tracked separately.

**Action taken in `03_ingest_epc`**: a derived `fuel_category` column collapses all variants into five clean groups:

| Category | What it means in practice | Retrofit implication |
|---|---|---|
| `gas_individual` | Property has its own gas boiler | Upgrade boiler or replace with heat pump — can be done property by property |
| `gas_community` | Shared gas boiler serving multiple flats (e.g. a block) | Retrofit requires whole-block intervention; individual tenants cannot act independently |
| `electric_individual` | Property has individual electric heating (e.g. storage heaters) | Already off gas — upgrade path is typically modern heat pump or upgraded storage heaters |
| `electric_community` | Shared electric heating system | Same whole-block constraint as gas community |
| `other` | LPG, oil, biomass, biodiesel, unknowns | Often rural or edge-case properties; LPG/oil are costly to run and priority targets for electrification |

The ~75% on `gas_individual` are the most tractable for a standard heat pump programme. The ~13% on `gas_community` require a fundamentally different (and more complex) programme design.

## Summary of findings

- **42.4% of London social rented stock is below EPC C** — the 2030 government target. This is the headline figure.
- The `epc_raw/` folder contains both `certificates-*.csv` and `recommendations-*.csv` — must read only certificates here.
- Column names in raw files are all **lowercase** (e.g. `current_energy_rating` not `CURRENT_ENERGY_RATING`).
- `construction_age_band` is the best proxy for retrofit difficulty — low null rate and directly indicates wall type.
- `co2_emiss_curr_per_floor_area` is preferred over raw CO2 as it normalises for property size.
- Some certificates are 10+ years old — the dataset reflects condition at time of inspection.
- Borough is in `local_authority_label`, clean with no nulls — reliable join key for the gold layer.